# TP 1: Text Preprocessing
**Natural Language Processing**

### Objectives
- Clean raw text with plain Python, regular expressions and BeautifulSoup.
- Tokenize text into words, sentences, Arabic units and BPE subwords.
- Normalize text: contractions, accents, Arabic letters, elongation, stop words.
- Build a preprocessing pipeline adapted to the model that will use the text.

### Instructions
- The work is individual. AI assistants (ChatGPT, Claude, etc.) are not allowed.
- In each exercise, write your function in the cell marked `# YOUR CODE HERE`, run it, then run the check cell below it. When a test fails, the check cell prints the input, the expected output and your output.
- Read the examples given with each exercise before writing code. Test your function on them first.
- Hints are given under the exercises. Open them only after you have tried.
- Questions marked **Q** are answered in the *Answer* cells.
- `progress()` lists the exercises you have validated.

**To submit:** this notebook, with every cell executed and every question answered.

## Part 0. Setup
Run the two cells below. The second one loads the data and the check functions. Do not modify it.

In [39]:
!pip install -q nltk beautifulsoup4
import nltk
for pkg in ["punkt", "punkt_tab", "wordnet"]:
    nltk.download(pkg, quiet=True)


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [40]:
# ========================= DO NOT MODIFY THIS CELL =========================
import re, unicodedata, warnings
from collections import Counter
warnings.filterwarnings("ignore", module="bs4")

# Dataset: 12 comments (invented for this TP)
COMMENTS = [
    "<p>OMG!!! The new phone is <b>AMAZING</b> 😍😍 #Tech @tech_dz https://t.co/x7Yz</p>",
    "Delivery was sooooo slow... I'm NOT happy :( https://shop.dz/order/123",
    "I can't believe it isn't working!!! 2 days without internet 😡 @telco_help",
    "Great service, really. Waited 3 hours. Great. 🙄",
    "الخدمـــة ممتاااازة 😍 بصح التوصيل lent بزاف!!! @livraison_dz",
    "<div>Prix: 5000DA... c'est trop cher!!! Je n'achète plus 😤</div>",
    "Mail me at ali@univ.dz for the #NLP course, it's free!!!",
    "المنتج جيد جدا، أنصح به 👍 #جودة",
    "ما عجبنيش المنتج، ماشي كيما في الصورة 😞",
    "Best purchase of 2026!!! 10/10 would buy again 🔥🔥",
    "Le produit est arrivé cassé :( très déçu... http://bit.ly/3xYz",
    "Dr. Ali said the battery lasts 2 days. It lasted 2 hours.",
]

HTML_SAMPLE = "<p>Hello <b>World</b>!</p><script>alert('x')</script>"

# ASCII punctuation (same characters as string.punctuation)
PUNCT = "!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~"

ABBREVIATIONS = {"dr.", "mr.", "mrs.", "prof.", "a.m.", "p.m.", "e.g.", "i.e.", "etc."}

# Order matters: full forms first, then suffixes
CONTRACTIONS = {
    "can't": "cannot", "won't": "will not", "n't": " not", "'m": " am", "'re": " are",
    "'s": " is", "'ll": " will", "'ve": " have", "'d": " would",
}

STOPWORDS_EN = {"a", "an", "the", "is", "are", "was", "were", "be", "been", "am", "i", "you", "he", "she",
    "it", "we", "they", "me", "my", "this", "that", "these", "those", "of", "to", "in", "on", "at", "for",
    "with", "and", "or", "but", "so", "very", "not", "no", "do", "does", "did", "have", "has", "had",
    "will", "would", "can", "just", "than", "then", "there", "here", "really"}
STOPWORDS_FR = {"le", "la", "les", "un", "une", "des", "de", "du", "et", "est", "je", "tu", "il", "elle",
    "nous", "vous", "ils", "ce", "c", "cette", "en", "au", "aux", "pour", "par", "sur", "très", "ne",
    "n", "pas", "plus", "que", "qui", "trop"}
STOPWORDS_AR = {"في", "من", "على", "إلى", "عن", "مع", "هذا", "هذه", "ذلك", "التي", "الذي", "هو", "هي",
    "كان", "ما", "لا", "لم", "لن", "ليس", "كل", "قد", "ثم", "أو", "و", "يا", "جدا", "به"}
NEGATIONS = {"not", "no", "never", "nor", "ne", "n", "pas", "لا", "لم", "لن", "ليس", "ما", "ماشي", "مش"}

EMOJI_MAP = {"😍": " <POS> ", "👍": " <POS> ", "🔥": " <POS> ", ":)": " <POS> ",
             "😡": " <NEG> ", "😞": " <NEG> ", "😤": " <NEG> ", "🙄": " <NEG> ", ":(": " <NEG> "}

BPE_CORPUS = {"low": 5, "lower": 2, "newest": 6, "widest": 3}
AR_CORPUS = {"المدرسة": 4, "المدارس": 3, "مدرسة": 2, "الكتاب": 5, "كتاب": 2, "الطلاب": 4, "طلاب": 1}

# ---------------------------- check functions ----------------------------
_ALL = ["1.1", "1.2", "1.3", "1.4", "1.5", "2.1a", "2.1b", "2.1c", "2.1d", "2.1e", "2.1f", "2.2", "2.3a",
        "2.3b", "2.4", "2.5", "2.6", "3.1", "3.3", "3.4", "3.5a", "3.5b", "3.5c", "3.5d", "4.1", "4.2",
        "4.3", "4.4", "4.5", "5.1", "5.2", "5.3"]
_PASSED = set()

def _fmt(x):
    s = repr(x)
    return s if len(s) <= 300 else s[:300] + " ..."

def check(name, func, cases):
    """Runs func on each (arguments, expected) pair and stops at the first failure."""
    if func is None:
        print(f"Exercise {name}: function not defined. Run your code cell first."); return False
    for i, (args, expected) in enumerate(cases, 1):
        if not isinstance(args, tuple):
            args = (args,)
        shown = args[0] if len(args) == 1 else args
        try:
            got = func(*args)
        except Exception as e:
            print(f"Exercise {name}: FAILED, your function raised {type(e).__name__}: {e}")
            print(f"   input    : {_fmt(shown)}"); return False
        if got != expected:
            print(f"Exercise {name}: FAILED on test {i}/{len(cases)}")
            print(f"   input    : {_fmt(shown)}")
            print(f"   expected : {_fmt(expected)}")
            print(f"   obtained : {_fmt(got)}")
            if got is None:
                print("   (your function returns None: check the return statement)")
            return False
    _PASSED.add(name)
    print(f"Exercise {name}: OK ({len(cases)}/{len(cases)} tests)")
    return True

def check_pattern(name, pattern, should_match, should_not_match):
    """The pattern must match each string of should_match entirely, and none of should_not_match."""
    if not pattern:
        print(f"Exercise {name}: no pattern yet."); return False
    try:
        rx = re.compile(pattern)
    except re.error as e:
        print(f"Exercise {name}: FAILED, invalid regular expression ({e})"); return False
    for s in should_match:
        if not rx.fullmatch(s):
            print(f"Exercise {name}: FAILED, the pattern must match the whole string {s!r}"); return False
    for s in should_not_match:
        if rx.fullmatch(s):
            print(f"Exercise {name}: FAILED, the pattern must not match {s!r}"); return False
    _PASSED.add(name)
    print(f"Exercise {name}: OK")
    return True

def check_true(name, conditions):
    """conditions: list of (boolean, message printed if the boolean is False)."""
    for ok, msg in conditions:
        if not ok:
            print(f"Exercise {name}: FAILED, {msg}"); return False
    _PASSED.add(name)
    print(f"Exercise {name}: OK")
    return True

def progress():
    done = [e for e in _ALL if e in _PASSED]
    todo = [e for e in _ALL if e not in _PASSED]
    print(f"Validated: {len(done)}/{len(_ALL)}")
    if todo:
        print("Remaining:", ", ".join(todo))

print("Setup done:", len(COMMENTS), "comments loaded.")

Setup done: 12 comments loaded.


## Part 1. Cleaning with plain Python

In this part, do not use `import` or regular expressions: only loops, conditions and string methods. The goal is to see what the libraries of the next parts do for us.

Python functions you will need:

| Operation | Python |
|---|---|
| loop over the characters of a string | `for ch in text:` |
| test if a character belongs to a string | `ch in "!?"` |
| build a string step by step | `result = ""` then `result += ch` |
| lowercase | `text.lower()` |
| test if a character is a digit | `ch.isdigit()` |
| split on spaces, tabs and new lines | `text.split()` returns a list of words |
| join a list with spaces | `" ".join(words)` |
| count with a dictionary | `counts[w] = counts.get(w, 0) + 1` |

Example of the cleaning we want: `"Hi!!! Are you coming??? $$$ 123"` becomes `"hi are you coming"`.

### Exercise 1.1. Remove a set of characters
Write `remove_chars(text, chars)` which returns `text` without the characters that appear in `chars`.

Example: `remove_chars("Hi!!! Are you coming???", "!?")` returns `"Hi Are you coming"`.

In [41]:
def remove_chars(text, chars):
    # YOUR CODE HERE
    result = ""
    for ch in text: 
        if ch not in chars :
            result += ch 
    
    return result
        
    

print(f"test 1: {remove_chars("Hi!!! Are you coming???", "!?")} \ntest 2: {remove_chars("$$$ 123 ok", "$")} \ntest 3: {remove_chars("abc", "")} \ntest 4: {remove_chars("", "!")}")

test 1: Hi Are you coming 
test 2:  123 ok 
test 3: abc 
test 4: 


In [42]:
check("1.1", remove_chars, [
    (("Hi!!! Are you coming???", "!?"), "Hi Are you coming"),
    (("$$$ 123 ok", "$"), " 123 ok"),
    (("abc", ""), "abc"),
    (("", "!"), ""),
])

Exercise 1.1: OK (4/4 tests)


True

<details><summary>Hint 1</summary>

Start from `result = ""`. Go through the characters of `text` and add a character to `result` only if it is not in `chars`.

</details>

<details><summary>Hint 2</summary>

```python
for ch in text:
    if ch not in chars:
        result += ch
return result
```

</details>

### Exercise 1.2. Remove digits
Write `remove_digits(text)` which removes every digit.

Example: `remove_digits("Version2 of the model: 95%")` returns `"Version of the model: %"`.

In [43]:
def remove_digits(text):
    result = "" 
    for ch in text:
        if not ch.isdigit():
            result += ch 
    return result
    


print(f"test 1: {remove_digits("Version2 of the model: 95%")} \ntest 2: {remove_digits("٢٠٢٦ is 2026")} \ntest 3: {remove_digits("no digits")}")

test 1: Version of the model: % 
test 2:  is  
test 3: no digits


In [44]:
check("1.2", remove_digits, [
    ("Version2 of the model: 95%", "Version of the model: %"),
    ("٢٠٢٦ is 2026", " is "),
    ("no digits", "no digits"),
])

Exercise 1.2: OK (3/3 tests)


True

<details><summary>Hint 1</summary>

Same structure as 1.1. The condition is `ch.isdigit()`.

</details>

<details><summary>Hint 2</summary>

Test 2 contains Arabic-Indic digits (٢٠٢٦). `isdigit()` also returns `True` for them, so you have nothing special to do.

</details>

### Exercise 1.3. Normalize whitespace
Write `normalize_spaces(text)` which replaces every sequence of spaces, tabs (`\t`) and new lines (`\n`) by a single space, and removes the spaces at the beginning and at the end.

Example: `normalize_spaces("  Hello \t\t world \n\n NLP  ")` returns `"Hello world NLP"`.

In [45]:
def normalize_spaces(text):
    text = text.split()
    return " ".join(text)

print(f"test 1: {normalize_spaces("  Hello \t\t world \n\n NLP  ")} \ntest 2: {normalize_spaces("")} \ntest 3: {normalize_spaces("one")}") 

test 1: Hello world NLP 
test 2:  
test 3: one


In [46]:
check("1.3", normalize_spaces, [
    ("  Hello \t\t world \n\n NLP  ", "Hello world NLP"),
    ("", ""),
    ("one", "one"),
])

Exercise 1.3: OK (3/3 tests)


True

<details><summary>Hint 1</summary>

Look at the table at the top of Part 1: one method splits on any whitespace, another one joins a list.

</details>

<details><summary>Hint 2</summary>

`text.split()` (without argument) splits on any whitespace and ignores empty pieces. Join the result with `" ".join(...)`.

</details>

### Exercise 1.4. A first cleaning function
1. Define the string `SYMBOLS` with the non-ASCII symbols you want to remove (currencies, French quotation marks, Arabic punctuation...). `PUNCT`, which contains the ASCII punctuation, is defined in the setup cell.
2. Write `clean_pure(text)` which: converts the text to lowercase, removes the characters of `PUNCT` and `SYMBOLS`, removes digits, then normalizes whitespace.

Use the functions of exercises 1.1 to 1.3.

In [47]:
import string

SYMBOLS = "€£«»…،؟"
PUNCT = string.punctuation
chars = SYMBOLS + PUNCT

def clean_pure(text):
    text = text.lower()
    text = remove_chars(text,  chars)
    text = remove_digits(text)
    text = normalize_spaces(text)
    return text 


print(f"test1: {clean_pure("Hi!!! Are you coming??? $$$ 123")} \ntest 2: {clean_pure("Prix: 5000€ «très» cher…")} \ntest 3: {clean_pure("مرحبا، كيف حالك؟")} \ntest 4: {clean_pure("Tab\tand\nnewline")}")

test1: hi are you coming 
test 2: prix très cher 
test 3: مرحبا كيف حالك 
test 4: tab and newline


In [48]:
check("1.4", clean_pure, [
    ("Hi!!! Are you coming??? $$$ 123", "hi are you coming"),
    ("Prix: 5000€ «très» cher…", "prix très cher"),
    ("مرحبا، كيف حالك؟", "مرحبا كيف حالك"),
    ("Tab\tand\nnewline", "tab and newline"),
])

Exercise 1.4: OK (4/4 tests)


True

<details><summary>Hint 1</summary>

Call your functions one after the other: `text = text.lower()`, then `remove_chars(text, PUNCT + SYMBOLS)`, then `remove_digits`, then `normalize_spaces`.

</details>

<details><summary>Hint 2</summary>

When a test fails, compare "expected" and "obtained": the characters that remain must be added to `SYMBOLS`. The Arabic comma `،` and question mark `؟` are not in `PUNCT`, because `PUNCT` only contains ASCII characters.

</details>

### Exercise 1.5. Count words
Write `word_counts(text)` which returns a dictionary `{word: number of occurrences}`. Use `text.split()` to get the words.

Example: `word_counts("the cat and the dog")` returns `{"the": 2, "cat": 1, "and": 1, "dog": 1}`.

The number of keys of this dictionary is the vocabulary size of the text. It is used again in Part 5.

In [93]:
def word_counts(text):
    counts = {}
    for word in text.split():
        counts[word] = counts.get(word, 0) + 1
    return counts
    

print(f"test 1: {word_counts("the cat and the dog")} \ntest 2: {word_counts("")}\ntest 3: {word_counts("NLP nlp NLP")}")


test 1: {'the': 2, 'cat': 1, 'and': 1, 'dog': 1} 
test 2: {}
test 3: {'NLP': 2, 'nlp': 1}


In [50]:
check("1.5", word_counts, [
    ("the cat and the dog", {"the": 2, "cat": 1, "and": 1, "dog": 1}),
    ("", {}),
    ("NLP nlp NLP", {"NLP": 2, "nlp": 1}),
])

Exercise 1.5: OK (3/3 tests)


True

<details><summary>Hint 1</summary>

Start with `counts = {}` and go through `text.split()`.

</details>

<details><summary>Hint 2</summary>

`counts[w] = counts.get(w, 0) + 1` adds 1 to the count of `w`, starting from 0 the first time.

</details>

Run the cell below to compare the vocabulary size before and after `clean_pure` on the 12 comments.

In [51]:
if {"1.4", "1.5"} <= _PASSED:
    raw_vocab = word_counts(" ".join(COMMENTS))
    clean_vocab = word_counts(" ".join(clean_pure(c) for c in COMMENTS))
    print("Vocabulary size: raw =", len(raw_vocab), "| after clean_pure =", len(clean_vocab))
else:
    print("Validate exercises 1.4 and 1.5 first.")

Vocabulary size: raw = 106 | after clean_pure = 96


### Question 1.6. Is every symbol noise?
Consider the sentence `"He scored 88% and %% won the 1%st and 2nd game"`. The result we want is `"He scored 88% and won the 1st and 2nd game"`: the `%` of `88%` is meaningful, the other ones are noise.

Run the cell below, then answer.

**Q1.6.** What does `clean_pure` return? Why can a function like `remove_chars` not produce the result we want? Describe in a few words a rule that keeps only the meaningful `%`. (This rule is implemented in exercise 2.5.)

In [52]:
print(clean_pure("He scored 88% and %% won the 1%st and 2nd game"))

he scored and won the st and nd game


**Answer Q1.6:**
<br>
**1-clean_pure :** It returns "he scored and won the st and nd game". It deletes all the numbers and all the % signs completely.<br>
**2-remove_chars** looks at each character alone without checking the context around it. It either removes every single % or keeps every %. It cannot tell if a % belongs to a real number (like 88%) or if it is just noise (like %%)<br>
**3- the rule :** Keep  % only if it comes right after a number and is not followed by a letter or a digit. Delete all other % signs.<br>

## Part 2. Regular expressions and cleaning

From now on you can use the `re` module. A regular expression (regex) describes a pattern of characters. The table below contains everything needed in this TP.

| Pattern | Meaning | Example |
|---|---|---|
| `\d` | a digit | `\d\d` matches `"42"` |
| `\w` | a letter, a digit or `_` (Arabic letters included) | `\w+` matches `"NLP"`, `"جودة"` |
| `\s` / `\S` | a whitespace / any character except whitespace | `\S+` matches `"t.co/x7"` |
| `.` | any character (`\.` is a real dot) | `a.c` matches `"abc"` |
| `[abc]` / `[^abc]` | one of a, b, c / any character except a, b, c | `[^\w\s]` = punctuation |
| `+` `*` `?` | 1 or more / 0 or more / 0 or 1 (optional) | `https?` matches `http` and `https` |
| `{2,}` | 2 times or more | `o{2,}` matches `"oo"`, `"ooo"` |
| `(...)` and `\1` | a group, and a reference to what it matched | `(.)\1` matches `"ll"` |
| `(?:...)` | a group that is not captured | `(?:\.\d+)?` = optional decimal part |
| `(?<!x)` / `(?=x)` | not preceded by x / followed by x (x is not consumed) | `(?<!\d)%` = a `%` not after a digit |

| Function | Use |
|---|---|
| `re.sub(pattern, repl, text)` | replaces every match |
| `re.findall(pattern, text)` | list of the matches (of group 1 if the pattern has one group) |
| `re.fullmatch(pattern, s)` | tests whether the whole string matches |

The bar `|` means "or": `cat|dog` matches `cat` or `dog`. Write patterns as raw strings: `r"\d+"`. Lessons 1 to 9 of the site regexone.com are a good exercise if regular expressions are new to you.

### Exercise 2.1. Six patterns
Replace each `None` by a pattern. The check uses `re.fullmatch`, so the pattern must match the whole string. The strings to accept and to reject are listed in the check cell.

In [94]:
PAT_INT = r"\d+"  # a whole number
PAT_DEC =  r"\d+(\.\d+)?" # a number with an optional decimal part
PAT_HASHTAG = r"#\w+"  # a hashtag
PAT_MENTION =  r"@\w+"   # a mention
PAT_EMAIL =  r"[\w.]+@\w+(\.\w+)+"   # an email address
PAT_URL = r"https?://\S+|www\.\S+"   # a URL

In [95]:
check_pattern("2.1a", PAT_INT,
    ["7", "2026", "000"],
    ["3.5", "12a", "", "-4"])
check_pattern("2.1b", PAT_DEC,
    ["10", "3.14", "0.5"],
    ["3.", "abc", ".5", "1.2.3"])
check_pattern("2.1c", PAT_HASHTAG,
    ["#NLP", "#جودة", "#AI2026", "#machine_learning"],
    ["NLP", "#", "# NLP", "##NLP"])
check_pattern("2.1d", PAT_MENTION,
    ["@tech_dz", "@Ali"],
    ["@", "tech_dz", "@ tech"])
check_pattern("2.1e", PAT_EMAIL,
    ["ali@univ.dz", "a.b@mail.com", "sara_99@gmail.com"],
    ["ali@", "@univ.dz", "ali univ@dz.com", "ali@univ"])
check_pattern("2.1f", PAT_URL,
    ["https://t.co/x7Yz", "http://bit.ly/3xYz", "www.nlp.dz/course"],
    ["htp://x.com", "https//x.com", "nlp.dz"])

Exercise 2.1a: OK
Exercise 2.1b: OK
Exercise 2.1c: OK
Exercise 2.1d: OK
Exercise 2.1e: OK
Exercise 2.1f: OK


True

<details><summary>Hint 1</summary>

2.1a: one or more digits. 2.1b: a whole number, then optionally a dot followed by digits. Put the optional part in a group followed by `?`, and write the dot `\.`.

</details>

<details><summary>Hint 2</summary>

2.1c and 2.1d: a symbol followed by one or more word characters. 2.1e: letters, digits or dots, then `@`, a word, then one or more groups "dot + word". 2.1f: `http` with an optional `s`, then `://`, then characters that are not spaces; or `www\.` followed by characters that are not spaces.

</details>

<details><summary>Hint 3</summary>

Shapes of the answers: `r"#\w+"`, `r"\d+(\.\d+)?"`, `r"[\w.]+@\w+(\.\w+)+"`. For the URL, put two alternatives on each side of `|`.

</details>

### Exercise 2.2. Extract hashtags
Write `extract_hashtags(text)` which returns the list of the hashtag words, without `#`.

Example: `extract_hashtags("Loving #NLP and #جودة! #AI2026")` returns `["NLP", "جودة", "AI2026"]`.

In [171]:
def extract_hashtags(text):
    return re.findall(r"#(\w+)", text)


print(f"test 1 : {extract_hashtags("Loving #NLP and #جودة! #AI2026")}\ntest 2: {extract_hashtags("no tags here")} \ntest 3: {extract_hashtags(COMMENTS[6])}")

test 1 : ['NLP', 'جودة', 'AI2026']
test 2: [] 
test 3: ['NLP']


In [134]:
check("2.2", extract_hashtags, [
    ("Loving #NLP and #جودة! #AI2026", ["NLP", "جودة", "AI2026"]),
    ("no tags here", []),
    (COMMENTS[6], ["NLP"]),
])

Exercise 2.2: OK (3/3 tests)


True

<details><summary>Hint 1</summary>

Use `re.findall`. When the pattern contains one group `( )`, `findall` returns only the content of the group.

</details>

<details><summary>Hint 2</summary>

Put `#` outside the group and `\w+` inside: `r"#(\w+)"`.

</details>

### Exercise 2.3. Remove HTML tags: regex or BeautifulSoup
Text collected on the web contains HTML tags (`<p>`, `<b>`...) and sometimes JavaScript code inside `<script>` tags.

a) Write `remove_html_regex(html)` which deletes every tag with the pattern `<.*?>`. The `?` makes the pattern lazy: it stops at the first `>`.

b) Write `remove_html_bs4(html)` with BeautifulSoup. BeautifulSoup reads the HTML as a tree of tags:
- `soup = BeautifulSoup(html, "html.parser")` builds the tree;
- `soup(["script", "style"])` returns the list of the `script` and `style` tags, and `tag.decompose()` deletes a tag with its content;
- `soup.get_text(" ")` returns the text with a space between the pieces.

Normalize the whitespace at the end.

**Q2.3.** Compare the outputs of the two functions on `HTML_SAMPLE`. Then try the greedy pattern `<.*>` (without `?`) on `HTML_SAMPLE`: what does it return, and why?

In [214]:
from bs4 import BeautifulSoup

def remove_html_regex(html):
    return re.sub(r"<.*?>", "", html)

def remove_html_bs4(html):
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style"]):
        tag.decompose()
    return " ".join(soup.get_text(" ").split())





print(f"test 1 : {remove_html_regex('<p>Hello <b>World</b>!</p>')}")
print(f"test 2 : {remove_html_regex(HTML_SAMPLE)}")
print(f"test 3 : {remove_html_bs4(HTML_SAMPLE)}")
print(f"test 4 : {remove_html_bs4("<p>Hi</p><p>there</p>")}")
print(f"test 5 : {remove_html_bs4('<style>p{color:red}</style><div>NLP</div>')}")


test 1 : Hello World!
test 2 : Hello World!alert('x')
test 3 : Hello World !
test 4 : Hi there
test 5 : NLP


In [173]:
check("2.3a", remove_html_regex, [
    ("<p>Hello <b>World</b>!</p>", "Hello World!"),
    (HTML_SAMPLE, "Hello World!alert('x')"),
])
check("2.3b", remove_html_bs4, [
    (HTML_SAMPLE, "Hello World !"),
    ("<p>Hi</p><p>there</p>", "Hi there"),
    ("<style>p{color:red}</style><div>NLP</div>", "NLP"),
])

Exercise 2.3a: OK (2/2 tests)
Exercise 2.3b: OK (3/3 tests)


True

<details><summary>Hint 1</summary>

a) One line with `re.sub`. b) Build the tree, loop over `soup(["script", "style"])` and call `.decompose()` on each tag.

</details>

<details><summary>Hint 2</summary>

b) After the loop: `text = soup.get_text(" ")`, then normalize whitespace as in exercise 1.3. The space before "!" in "Hello World !" comes from the separator; it does no harm, since tokenization separates punctuation anyway.

</details>

**Answer Q2.3:**


**1. Comparison on HTML_SAMPLE:**

- `remove_html_regex` returns: `"Hello World!alert('x')"`. It only deletes the tags, but leaves behind the JavaScript code from inside the `<script>` tag.

- `remove_html_bs4` returns: `"Hello World !"`. It removes the `<script>` tag together with the code inside it, keeping only the visible text.

**2. Greedy pattern `<.*>`:**

- It returns an empty string `""`.

- **Why:** The pattern `.*` is greedy, meaning it takes as many characters as possible. It starts from the first `<` in `<p>` and goes all the way to the last `>` at the end of `</script>`, deleting all the text in between.



### Exercise 2.4. Emails, URLs, mentions and hashtags
Write `replace_entities(text)` which, in this order:
1. replaces the emails by `<EMAIL>`;
2. replaces the URLs (`http://`, `https://` or `www.`) by `<URL>`;
3. replaces the mentions by `<USER>`;
4. replaces `#word` by `word`.

Reuse your patterns of exercise 2.1.

**Q2.4.** Why do we replace URLs by `<URL>` instead of deleting them? What happens if step 3 is done before step 1 on `"ali@univ.dz"`?

In [217]:
def replace_entities(text):
    text = re.sub(PAT_EMAIL, "<EMAIL>", text)
    text = re.sub(PAT_URL, "<URL>", text)
    text = re.sub(PAT_MENTION, "<USER>", text)
    text = re.sub(r"#(\w+)", r"\1", text)
    return text




print(f"test 1: {replace_entities('Mail ali@univ.dz or see https://nlp.dz/course @prof_ali #NLP')}")
print(f"test 2: {replace_entities('Visit www.nlp.dz now')}")
print(f"test 3: {replace_entities('Thanks @ali_99 for #جودة')}")


test 1: Mail <EMAIL> or see <URL> <USER> NLP
test 2: Visit <URL> now
test 3: Thanks <USER> for جودة


In [218]:
check("2.4", replace_entities, [
    ("Mail ali@univ.dz or see https://nlp.dz/course @prof_ali #NLP", "Mail <EMAIL> or see <URL> <USER> NLP"),
    ("Visit www.nlp.dz now", "Visit <URL> now"),
    ("Thanks @ali_99 for #جودة", "Thanks <USER> for جودة"),
])

Exercise 2.4: OK (3/3 tests)


True

<details><summary>Hint 1</summary>

Four `re.sub` lines, one per step. In the replacement string, `\1` stands for what the first group matched: `re.sub(r"#(\w+)", r"\1", text)`.

</details>

<details><summary>Hint 2</summary>

If you get `ali<USER>.dz`, the mention step was applied before the email step, or your email pattern does not match.

</details>

**Answer Q2.4:**

**1. Why replace URLs with `<URL>` instead of deleting them?**

- Deleting URLs removes the information that a link was shared.
- Replacing them with `<URL>` preserves the context of the sentence while preventing the vocabulary from filling up with thousands of unique web addresses.

**2. What happens if step 3 (mentions) is done before step 1 (emails)?**

- In `"ali@univ.dz"`, the `@univ` part looks like a mention.
- If step 3 runs first, it replaces `@univ` with `<USER>`, turning the text into `"ali<USER>.dz"`.
- As a result, the email pattern will no longer recognize it as an email address, corrupting the data.


### Exercise 2.5. The `%` sign
Back to question 1.6. Write `clean_percent(s)` which keeps a `%` only if it comes right after a digit and is not followed by a letter or a digit. The other `%` are deleted, then whitespace is normalized.

Example: `"He scored 88% and %% won the 1%st and 2nd game"` gives `"He scored 88% and won the 1st and 2nd game"`.

In [281]:
def clean_percent(s):
    s = re.sub(r"(?<!\d)%|%(?=\w)", "", s)
    return " ".join(s.split())




print(f"test 1: {clean_percent('He scored 88% and %% won the 1%st and 2nd game')}")
print(f"test 2: {clean_percent('50% off today')}")
print(f"test 3: {clean_percent('100%% sure')}")
print(f"test 4: {clean_percent('%%% wow')}")

test 1: He scored 88% and won the 1st and 2nd game
test 2: 50% off today
test 3: 100% sure
test 4: wow


In [251]:
check("2.5", clean_percent, [
    ("He scored 88% and %% won the 1%st and 2nd game", "He scored 88% and won the 1st and 2nd game"),
    ("50% off today", "50% off today"),
    ("100%% sure", "100% sure"),
    ("%%% wow", "wow"),
])

Exercise 2.5: OK (4/4 tests)


True

<details><summary>Hint 1</summary>

Describe the `%` to delete. There are two cases: a `%` that is not preceded by a digit, and a `%` followed by a word character. See the last line of the first table of Part 2.

</details>

<details><summary>Hint 2</summary>

`(?<!\d)%` matches a `%` that does not follow a digit. `%(?=\w)` matches a `%` followed by a letter or a digit, without taking that letter. Join the two with `|` in a single `re.sub`, then normalize whitespace.

</details>

### Exercise 2.6. The function `clean_text`
Write `clean_text(text)`: `remove_html_bs4`, then `replace_entities`, then normalization of whitespace.

`clean_text` does not change the case, the punctuation or the emojis: whether to keep them depends on the model (Part 5).

In [282]:
def clean_text(text):
    text = remove_html_bs4(text)
    text = replace_entities(text)
    return " ".join(text.split())


In [283]:
check("2.6", clean_text, [
    (COMMENTS[0], "OMG!!! The new phone is AMAZING 😍😍 Tech <USER> <URL>"),
    ("<p>Hi</p><p>there</p>", "Hi there"),
    (COMMENTS[6], "Mail me at <EMAIL> for the NLP course, it's free!!!"),
])
for c in COMMENTS:
    print(clean_text(c))

Exercise 2.6: OK (3/3 tests)
OMG!!! The new phone is AMAZING 😍😍 Tech <USER> <URL>
Delivery was sooooo slow... I'm NOT happy :( <URL>
I can't believe it isn't working!!! 2 days without internet 😡 <USER>
Great service, really. Waited 3 hours. Great. 🙄
الخدمـــة ممتاااازة 😍 بصح التوصيل lent بزاف!!! <USER>
Prix: 5000DA... c'est trop cher!!! Je n'achète plus 😤
Mail me at <EMAIL> for the NLP course, it's free!!!
المنتج جيد جدا، أنصح به 👍 جودة
ما عجبنيش المنتج، ماشي كيما في الصورة 😞
Best purchase of 2026!!! 10/10 would buy again 🔥🔥
Le produit est arrivé cassé :( très déçu... <URL>
Dr. Ali said the battery lasts 2 days. It lasted 2 hours.


## Part 3. Tokenization

Tokenization splits a text into units called tokens. Depending on the need, a token can be a sentence, a word, a part of a word (subword) or a character. The set of distinct tokens known by a model is its vocabulary.

Splitting on spaces is not enough: in `"this!"` the punctuation is attached to the word, `"3.50"` contains a dot that does not end a sentence, and the Arabic word `"وبالمدرسة"` contains three units.

### Exercise 3.1. A word tokenizer
Write `tokenize(text)` which returns the list of tokens with a single `re.findall`. The pattern is made of four alternatives separated by `|`. The order is important: at each position, the first alternative that matches is used.

1. a placeholder such as `<URL>` or `<user>` is one token;
2. a decimal number such as `3.50` is one token;
3. a word, with possible internal apostrophes, such as `can't` or `2nd`;
4. any other single character that is not a space (punctuation, `$`, emoji...).

Example: `tokenize("I can't pay $3.50 for this!")` returns `["I", "can't", "pay", "$", "3.50", "for", "this", "!"]`.

In [341]:
def tokenize(text):
    pattern = r"<[A-Za-z_]+>|\d+(?:\.\d+)+|\w+(?:'\w+)*|[^\w\s]"
    return re.findall(pattern, text)



print(f"test 1: {tokenize("I can't pay $3.50 for this!")}")
print(f"test 2: {tokenize('Hello, world!!!')}")
print(f"test 3: {tokenize('مرحبا، كيف حالك؟')}")
print(f"test 4: {tokenize('see <URL> now')}")
print(f"test 5: {tokenize('the 2nd game')}")


test 1: ['I', "can't", 'pay', '$', '3.50', 'for', 'this', '!']
test 2: ['Hello', ',', 'world', '!', '!', '!']
test 3: ['مرحبا', '،', 'كيف', 'حالك', '؟']
test 4: ['see', '<URL>', 'now']
test 5: ['the', '2nd', 'game']


In [312]:
check("3.1", tokenize, [
    ("I can't pay $3.50 for this!", ["I", "can't", "pay", "$", "3.50", "for", "this", "!"]),
    ("Hello, world!!!", ["Hello", ",", "world", "!", "!", "!"]),
    ("مرحبا، كيف حالك؟", ["مرحبا", "،", "كيف", "حالك", "؟"]),
    ("see <URL> now", ["see", "<URL>", "now"]),
    ("the 2nd game", ["the", "2nd", "game"]),
])

Exercise 3.1: OK (5/5 tests)


True

<details><summary>Hint 1</summary>

Write the four patterns separately and test each one with `re.findall` before joining them: a placeholder `<[A-Za-z_]+>`; a decimal number `\d+(?:\.\d+)+`; a word `\w+` followed by zero or more groups "apostrophe + letters"; a character that is neither a word character nor a space.

</details>

<details><summary>Hint 2</summary>

Full pattern: `r"<[A-Za-z_]+>|\d+(?:\.\d+)+|\w+(?:'\w+)*|[^\w\s]"`. Swap the 2nd and 3rd alternatives and test `"3.50"` to see why the order matters.

</details>

### Question 3.2. Comparison with `split()` and NLTK
Run the cell below.

**Q3.2.** Give two differences between your tokenizer and `word_tokenize` of NLTK. How does NLTK split `"can't"` and `"Don't"`? What is the interest of this choice for the word `not`?

In [367]:
from nltk.tokenize import word_tokenize
for s in ["I can't pay $3.50 for this!", "Don't stop, Mr. Smith!", "e-mail me at 10:30 😍"]:
    print("TEXT    :", s)
    print("split() :", s.split())
    print("yours   :", tokenize(s))
    print("NLTK    :", word_tokenize(s))
    print()

TEXT    : I can't pay $3.50 for this!
split() : ['I', "can't", 'pay', '$3.50', 'for', 'this!']
yours   : ['I', "can't", 'pay', '$', '3.50', 'for', 'this', '!']
NLTK    : ['I', 'ca', "n't", 'pay', '$', '3.50', 'for', 'this', '!']

TEXT    : Don't stop, Mr. Smith!
split() : ["Don't", 'stop,', 'Mr.', 'Smith!']
yours   : ["Don't", 'stop', ',', 'Mr', '.', 'Smith', '!']
NLTK    : ['Do', "n't", 'stop', ',', 'Mr.', 'Smith', '!']

TEXT    : e-mail me at 10:30 😍
split() : ['e-mail', 'me', 'at', '10:30', '😍']
yours   : ['e', '-', 'mail', 'me', 'at', '10', ':', '30', '😍']
NLTK    : ['e-mail', 'me', 'at', '10:30', '😍']



**Answer Q3.2:**

1-Two differences:

Our tokenizer keeps words like "can't" together in one piece, while NLTK breaks them apart into ca and n't.
NLTK understands abbreviations like "Mr." and words with hyphens like "e-mail", while our code cuts them into separate characters.


2-How NLTK splits them:

"can't" becomes ['ca', "n't"] <br>
"Don't" becomes ['Do', "n't"]


3-Why this helps with the word not:

Because "n't" just means "not". <br>
By isolating "n't", the model can easily detect negation in the text, which is super important for understanding sentiment (like knowing the difference between "happy" and "not happy").

### Exercise 3.3. Splitting into sentences
A period does not always end a sentence: in `"Dr. Ali arrived at 9 a.m. today."` only the last one does. Write `split_sentences(text, abbreviations=ABBREVIATIONS)`:
- cut the text into chunks with `text.split()`;
- add the chunks one by one to the current sentence;
- a chunk ends the sentence when its last character is `.`, `!`, `?` or `؟` and its lowercase form is not in `abbreviations`;
- return the list of sentences, the chunks of a sentence being joined by a space. The last sentence may have no final punctuation.

**Q3.3.** Find a sentence on which your function makes a mistake, and explain why.

In [368]:
def split_sentences(text, abbreviations=ABBREVIATIONS):
    sentences = []
    current = []
    
    for chunk in text.split():
        current.append(chunk)
        if chunk[-1] in {".", "!", "?", "؟"} and chunk.lower() not in abbreviations:
            sentences.append(" ".join(current))
            current = []
            
    if current:
        sentences.append(" ".join(current))
        
    return sentences


In [369]:
check("3.3", split_sentences, [
    ("Dr. Ali arrived at 9 a.m. today. The lab was closed.", ["Dr. Ali arrived at 9 a.m. today.", "The lab was closed."]),
    ("Hello! How are you? I love NLP.", ["Hello!", "How are you?", "I love NLP."]),
    ("أحب البرمجة. هل تحبها؟ نعم", ["أحب البرمجة.", "هل تحبها؟", "نعم"]),
    ("", []),
])
print(split_sentences(COMMENTS[11]))

Exercise 3.3: OK (4/4 tests)
['Dr. Ali said the battery lasts 2 days.', 'It lasted 2 hours.']


<details><summary>Hint 1</summary>

Use two lists: `sentences` for the finished sentences and `current` for the chunks of the sentence being built. `chunk[-1]` is the last character of a chunk.

</details>

<details><summary>Hint 2</summary>

After the loop, if `current` is not empty, join it and add it to `sentences` (case of "نعم").

</details>

**Answer Q3.3:**

exemple : I live in the U.S. today.  <br>


Why it fails: <br>
The word "U.S." ends with a dot (.), but "u.s." is not in our list of known abbreviations. Because of this, the function wrongly cuts the sentence in the middle and produces two broken pieces:
['I live in the U.S.', 'today.'].



### Exercise 3.4. Arabic clitics
In Arabic, the conjunctions (و، ف), some prepositions (ب، ك، ل) and the article (ال) are attached to the word. They are called clitics: `وبالمدرسة` = `و` (and) + `ب` (in) + `ال` (the) + `مدرسة` (school). When `ل` is followed by the article, the alef of the article is not written: `ل` + `الطلاب` is written `للطلاب`.

Write `segment_ar(word)` which applies the following rules in this order and returns the list of parts:
1. Conjunction: if the word starts with `و` or `ف` and at least 3 letters remain after it, separate it.
2. `لل`: if what remains starts with `لل` and has more than 3 letters, produce `ل` and `ال`, and keep the rest.
3. Otherwise, preposition: if what remains starts with `ب`, `ك` or `ل` and the two next letters are `ال`, separate the preposition.
4. Article: if what remains starts with `ال` and at least 2 letters remain after it, separate `ال`.

**Q3.4.** Look at the results printed for `وزير` (minister) and `والد` (father). What is wrong? What would a real tool need to avoid this error?

In [393]:
def segment_ar(word):
    parts = []
    
    
    if word and word[0] in "وف" and len(word[1:]) >= 3:
        parts.append(word[0])
        word = word[1:]
        
   
    if word.startswith("لل") and len(word) > 3:
        parts.append("ل")
        parts.append("ال")
        word = word[2:]
  
    elif word and word[0] in "بكل" and word[1:].startswith("ال"):
        parts.append(word[0])
        word = word[1:]
        
    
    if word.startswith("ال") and len(word[2:]) >= 2:
        parts.append("ال")
        word = word[2:]
        
   
    if word:
        parts.append(word)
        
    return parts


In [394]:
check("3.4", segment_ar, [
    ("وبالمدرسة", ["و", "ب", "ال", "مدرسة"]),
    ("للطلاب", ["ل", "ال", "طلاب"]),
    ("وللطلاب", ["و", "ل", "ال", "طلاب"]),
    ("فالكتاب", ["ف", "ال", "كتاب"]),
    ("المدرسة", ["ال", "مدرسة"]),
    ("باب", ["باب"]),
    ("كتب", ["كتب"]),
])
for w in ["وزير", "والد", "بالقلم"]:
    print(w, "->", segment_ar(w))

Exercise 3.4: OK (7/7 tests)
وزير -> ['و', 'زير']
والد -> ['و', 'الد']
بالقلم -> ['ب', 'ال', 'قلم']


<details><summary>Hint 1</summary>

Keep a variable `word` that you shorten at each rule: `parts.append(word[0]); word = word[1:]`. Rules 2 and 3 form an `if` / `elif`: only one of them can apply.

</details>

<details><summary>Hint 2</summary>

Rule 3: `word[0] in "بكل" and word[1:].startswith("ال")`. Rule 4: `word.startswith("ال") and len(word) - 2 >= 2`. At the end, add what remains of `word` to `parts`.

</details>

**Answer Q3.4:**

1-What is wrong: <br>
In both وزير (minister) and والد (father), the initial و is an original root letter of the word, not the conjunction "and" (و). The function mistakenly cuts off the و, breaking the words into nonsensical pieces (['و', 'زير'] and ['و', 'الد']).

2-What a real tool needs: <br>
A real morphological tool (like Farasa or CAMeL Tools) needs a dictionary (lexicon) of valid Arabic roots and words, along with context / Part-of-Speech tagging, so it only separates و when the remaining word is a recognized stem and when it grammatically functions as a conjunction.

### Exercise 3.5. Byte-Pair Encoding (BPE)
BPE builds a vocabulary of subwords from a corpus. It is used by the GPT models.

Training:
1. each word is split into characters, followed by the end-of-word symbol `</w>`: `"low"` gives `("l", "o", "w", "</w>")`;
2. count the pairs of adjacent symbols, each word counting as many times as it appears in the corpus;
3. merge the most frequent pair everywhere it appears: it becomes a new symbol;
4. repeat steps 2 and 3 a fixed number of times. The list of merges, in order, is the result of the training.

To tokenize a new word, split it into characters + `</w>` and apply the learned merges in the same order.

Example with `BPE_CORPUS = {"low": 5, "lower": 2, "newest": 6, "widest": 3}`: the pair `("e", "s")` appears 6 times in "newest" and 3 times in "widest", 9 times in all. It is the first merge.

Write the four functions:
- a) `get_pair_counts(words)`: `words` is a dictionary `{tuple of symbols: count}`. Return a `Counter` `{(a, b): total count}` of the adjacent pairs.
- b) `merge_pair(word, pair)`: return a new tuple in which every occurrence of the two adjacent symbols `pair` is replaced by their concatenation.
- c) `learn_bpe(corpus, n_merges)`: `corpus` is a dictionary `{word: count}`. Return the list of the `n_merges` merged pairs. In case of a tie, use `max(pairs, key=pairs.get)`, which returns the first pair found among the most frequent.
- d) `bpe_tokenize(word, merges)`: return the list of the symbols of `word` after applying the merges.

**Q3.5.** Run the last cell of this exercise. Which units does BPE learn first on the Arabic corpus `AR_CORPUS`? How is `بالمدرسة`, which is not in the corpus, tokenized?

In [415]:
def get_pair_counts(words):
    pairs = Counter()
    for word, count in words.items():
        for pair in zip(word, word[1:]):
            pairs[pair] += count
    return pairs

def merge_pair(word, pair):
    new_word = []
    i = 0
    while i < len(word):
        if i < len(word) - 1 and (word[i], word[i+1]) == pair:
            new_word.append(word[i] + word[i+1])
            i += 2
        else:
            new_word.append(word[i])
            i += 1
    return tuple(new_word)

def learn_bpe(corpus, n_merges):
    words = {tuple(w) + ("</w>",): c for w, c in corpus.items()}
    merges = []
    for _ in range(n_merges):
        pairs = get_pair_counts(words)
        if not pairs:
            break
        best = max(pairs, key=pairs.get)
        merges.append(best)
        words = {merge_pair(w, best): c for w, c in words.items()}
    return merges

def bpe_tokenize(word, merges):
    w = tuple(word) + ("</w>",)
    for pair in merges:
        w = merge_pair(w, pair)
    return list(w)


In [416]:
check("3.5a", get_pair_counts, [
    ({("l", "o", "w", "</w>"): 5, ("l", "o", "w", "e", "r", "</w>"): 2},
     {("l", "o"): 7, ("o", "w"): 7, ("w", "</w>"): 5, ("w", "e"): 2, ("e", "r"): 2, ("r", "</w>"): 2}),
])
check("3.5b", merge_pair, [
    ((("n", "e", "w", "e", "s", "t", "</w>"), ("e", "s")), ("n", "e", "w", "es", "t", "</w>")),
    ((("l", "o", "w", "</w>"), ("e", "s")), ("l", "o", "w", "</w>")),
    ((("a", "a", "a"), ("a", "a")), ("aa", "a")),
])
check("3.5c", learn_bpe, [
    ((BPE_CORPUS, 5), [("e", "s"), ("es", "t"), ("est", "</w>"), ("l", "o"), ("lo", "w")]),
])
MERGES = learn_bpe(BPE_CORPUS, 5)
check("3.5d", bpe_tokenize, [
    (("lowest", MERGES), ["low", "est</w>"]),
    (("slowest", MERGES), ["s", "low", "est</w>"]),
    (("newer", MERGES), ["n", "e", "w", "e", "r", "</w>"]),
])

Exercise 3.5a: OK (1/1 tests)
Exercise 3.5b: OK (3/3 tests)
Exercise 3.5c: OK (1/1 tests)
Exercise 3.5d: OK (3/3 tests)


True

<details><summary>Hint 1</summary>

a) For each word, `zip(word, word[1:])` gives the adjacent pairs. Add the count of the word (not 1) to each pair. b) Go through the tuple with an index `i` in a `while` loop: if `(word[i], word[i+1]) == pair`, add the concatenation and advance by 2; otherwise add `word[i]` and advance by 1.

</details>

<details><summary>Hint 2</summary>

c) Repeat `n_merges` times: count the pairs, take `best = max(pairs, key=pairs.get)`, add it to `merges`, then rebuild the dictionary with all the words merged: `words = {merge_pair(w, best): c for w, c in words.items()}`. d) Start from `tuple(word) + ("</w>",)` and call `merge_pair` for each pair of `merges`, in order.

</details>

In [417]:
ar_merges = learn_bpe(AR_CORPUS, 10)
for i, pair in enumerate(ar_merges, 1):
    print(i, pair, "->", pair[0] + pair[1])
print(bpe_tokenize("بالمدرسة", ar_merges))

1 ('ا', 'ل') -> ال
2 ('ا', 'ب') -> اب
3 ('اب', '</w>') -> اب</w>
4 ('م', 'د') -> مد
5 ('ر', 'س') -> رس
6 ('ال', 'مد') -> المد
7 ('ك', 'ت') -> كت
8 ('كت', 'اب</w>') -> كتاب</w>
9 ('رس', 'ة') -> رسة
10 ('رسة', '</w>') -> رسة</w>
['ب', 'المد', 'رسة</w>']


**Answer Q3.5:**

**1-Units learned first:**<br>
The very first merge learned is ('ا', 'ل') -> ال (the Arabic definite article "Al-"), because it appears across almost all words in the corpus. After that, it learns frequent word endings and stems like اب and رسة.

**2-How بالمدرسة is tokenized:**<br>
It is tokenized into: ['ب', 'المد', 'رسة</w>'].
Even though "بالمدرسة" was not in the training corpus, BPE does not fail with an "Out-Of-Vocabulary" error. It isolates the unseen letter 'ب' and reuses the known subword pieces 'المد' and 'رسة</w>'

## Part 4. Normalization

Normalization brings the different forms of the same word to a single form (`NLP`, `nlp`, `Nlp` give `nlp`). The vocabulary becomes smaller and each word is seen more often, but some information can be lost (`US` the country and `us` the pronoun become the same token).

### Exercise 4.1. Contractions
Write `expand_contractions(text, mapping=CONTRACTIONS)` which replaces each key of `mapping` by its value, in the order of the dictionary. `CONTRACTIONS` is defined in the setup cell: the full forms such as `can't` come before the suffixes such as `n't`.

Example: `expand_contractions("I can't believe it isn't working!")` returns `"I cannot believe it is not working!"`.

**Q4.1.** a) Look at the two first lines printed after the check. Why must contractions be expanded before removing punctuation? b) What is wrong with `"Ali's book"`?

In [480]:
def expand_contractions(text, mapping=CONTRACTIONS):
    for short, full in mapping.items():
        text = text.replace(short, full)
    return text


In [481]:
check("4.1", expand_contractions, [
    ("I can't believe it isn't working!", "I cannot believe it is not working!"),
    ("I'm sure you're right", "I am sure you are right"),
    ("It won't work", "It will not work"),
    ("Don't stop", "Do not stop"),
])
if "4.1" in _PASSED:
    text = "I can't believe it isn't working!"
    print("punctuation first :", re.sub(r"[^\w\s]", "", text).lower())
    print("contractions first:", re.sub(r"[^\w\s]", "", expand_contractions(text)).lower())
    print("possessive        :", expand_contractions("Ali's book is great"))

Exercise 4.1: OK (4/4 tests)
punctuation first : i cant believe it isnt working
contractions first: i cannot believe it is not working
possessive        : Ali is book is great


<details><summary>Hint 1</summary>

Loop over `mapping.items()` and use the method `text.replace(short, full)`. A Python dictionary keeps the order in which its keys were inserted.

</details>

**Answer Q4.1:**

**a) Why expand contractions before removing punctuation? **<br>
If we remove punctuation first, the apostrophe is lost: "can't" becomes "cant" and "isn't" becomes "isnt". Without the apostrophe, the mapping cannot find them anymore, so words like "not" are completely lost and replaced by broken words (cant, isnt).<br>

**b) What is wrong with "Ali's book"?**<br>
Here, 's is a possessive (showing ownership: the book of Ali), not the verb "is". Because the function replaces every 's with " is", it wrongly turns "Ali's book is great" into "Ali is book is great"

### Exercise 4.2. Accents
The letter `é` can be stored in two ways: one character, or `e` followed by an accent mark. The function `unicodedata.normalize("NFKD", text)` always uses the second form; it also splits ligatures such as `ﬁ` into `f` and `i`. The accent marks are then recognised by `unicodedata.combining(c)`, which is different from 0 for them.

Write `strip_accents(text)` which applies NFKD and keeps only the characters `c` for which `unicodedata.combining(c)` is 0.

**Q4.2.** What does `strip_accents` do to the Arabic sentence printed below? In which case is it a problem? Give a French example where removing accents changes the meaning.

In [497]:
def strip_accents(text):
    text = unicodedata.normalize("NFKD", text)
    return "".join(c for c in text if not unicodedata.combining(c))


In [498]:
check("4.2", strip_accents, [
    ("Café, naïve, Élève, ﬁle", "Cafe, naive, Eleve, file"),
    ("NLP", "NLP"),
])
print(strip_accents("أَحْمَد يَقْرَأُ"))

Exercise 4.2: OK (2/2 tests)
احمد يقرا


<details><summary>Hint 1</summary>

After the normalization: `"".join(c for c in text if not unicodedata.combining(c))`.

</details>

**Answer Q4.2:**

**1-What it does to the Arabic sentence:**<br>
It removes all the Arabic vocalization marks (tashkeel like fatha, damma, sukun) as well as the hamzas, turning "أَحْمَد يَقْرَأُ" into "احمد يقرا". This happens because Arabic diacritics are classified as combining characters in Unicode.
<br>
**2-When it becomes a problem:**<br>
It creates heavy ambiguity between words that share the exact same letters. For instance, without diacritics, عَلِمَ (he knew), عُلِمَ (it was known), and عِلْم (knowledge) all become علم, making it hard to distinguish active from passive voice.
<br>
**3-French example where removing accents changes meaning:**<br>

-ou (or) vs. où (where) <br>
-a (has) vs. à (at / to)<br>
-sur (on) vs. sûr (sure)

### Exercise 4.3. Arabic normalization
The same Arabic word is often written in several ways (with or without diacritics, أ or ا, ى or ي, ة or ه). Write `normalize_ar(text)` which applies the following rules in this order:

| Rule | Unicode | Example |
|---|---|---|
| remove the diacritics (tashkeel) | range `ً` to `ْ` | `المَدْرَسَةُ` -> `المدرسة` |
| remove the tatweel | `ـ` | `الكـــبيرة` -> `الكبيرة` |
| alef with hamza or madda -> alef | `آ أ إ` -> `ا` | `إلى` -> `الى` |
| alef maqsura -> ya | `ى` -> `ي` | `على` -> `علي` |
| ta marbuta -> ha | `ة` -> `ه` | `مدرسة` -> `مدرسه` |
| Arabic-Indic digits -> 0-9 | `٠١٢٣٤٥٦٧٨٩` | `٢٠٢٦` -> `2026` |

Write the letters with their `\u` codes in the patterns: Arabic letters written inside Python code are displayed right to left and are easy to confuse.

In [512]:
def normalize_ar(text):
    
    text = re.sub(r"[\u064B-\u0652]", "", text)
    
    text = re.sub(r"\u0640", "", text)
  
    text = re.sub(r"[\u0622\u0623\u0625]", "\u0627", text)
    
    text = re.sub(r"\u0649", "\u064A", text)
    
    text = re.sub(r"\u0629", "\u0647", text)
    
    text = text.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789"))
    
    return text


In [513]:
check("4.3", normalize_ar, [
    ("إلى المَدْرَسَةِ الكـــبيرة", "الي المدرسه الكبيره"),
    ("على", "علي"),
    ("آمال", "امال"),
    ("٢٠٢٦", "2026"),
    ("NLP 2026", "NLP 2026"),
])

Exercise 4.3: OK (5/5 tests)


True

<details><summary>Hint 1</summary>

Five `re.sub` and one `translate`. A range of characters is written between brackets: `r"[ً-ْ]"`. In the replacement, the ordinary string `"ا"` is the letter ا.

</details>

<details><summary>Hint 2</summary>

For the digits: `text.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789"))` replaces each Arabic-Indic digit by the digit at the same position in the second string.

</details>

### Exercise 4.4. Elongation
Letters are often repeated to express an emotion: `ممتاااازة`, `sooooo`. Write `reduce_elongation(text, keep=1)` which replaces a letter repeated 3 times or more by `keep` copies of this letter.
- Only letters are concerned: `1000 DA` must stay unchanged.
- Two identical letters are normal (`ممتازة` starts with مم, `good` contains oo): they are not changed.

In [525]:
def reduce_elongation(text, keep=1):
    pattern = r"([^\W\d_])\1{2,}"
    return re.sub(pattern, lambda m: m.group(1) * keep, text)


In [526]:
check("4.4", reduce_elongation, [
    ("ممتاااازة", "ممتازة"),
    ("ممتازة", "ممتازة"),
    ("sooooo good", "so good"),
    (("coooool", 2), "cool"),
    ("1000 DA!!!", "1000 DA!!!"),
])

Exercise 4.4: OK (5/5 tests)


True

<details><summary>Hint 1</summary>

A captured character followed by itself 2 times or more: `(.)\1{2,}`. But `.` also accepts digits and `!`. To accept letters only, use `[^\W\d_]`: neither a non-word character, nor a digit, nor `_`. This leaves the letters, Arabic included.

</details>

<details><summary>Hint 2</summary>

The replacement depends on `keep`: give a function to `re.sub`: `re.sub(pattern, lambda m: m.group(1) * keep, text)`. `m.group(1)` is the repeated letter.

</details>

### Exercise 4.5. Stop words
Stop words are very frequent words that say little about the subject of a text (the, is, في, من). Removing them reduces the data, but some of them are negations, which change the meaning of a sentence.

Write `remove_stopwords(tokens, stopwords, keep_negations=True)` which returns the tokens that are not stop words. If `keep_negations` is `True`, a token that belongs to `NEGATIONS` is always kept.

**Q4.5.** In which task is removing `not` a serious error? In which task is it harmless?

In [537]:
def remove_stopwords(tokens, stopwords, keep_negations=True):
    return [
        t for t in tokens
        if t not in stopwords or (keep_negations and t in NEGATIONS)
    ]


In [538]:
check("4.5", remove_stopwords, [
    ((["this", "movie", "is", "not", "good"], STOPWORDS_EN), ["movie", "not", "good"]),
    ((["this", "movie", "is", "not", "good"], STOPWORDS_EN, False), ["movie", "good"]),
    ((["هذا", "الفيلم", "ليس", "جيدا"], STOPWORDS_AR), ["الفيلم", "ليس", "جيدا"]),
])

Exercise 4.5: OK (3/3 tests)


True

<details><summary>Hint 1</summary>

A list comprehension with a condition: keep `t` if it is not a stop word, or if `keep_negations` is true and `t` is a negation.

</details>

**Answer Q4.5:**
<br>1-Where removing not is a serious error:<br>
Sentiment Analysis (opinion mining). Removing "not" reverses the entire polarity of the sentence: "not good" becomes "good", turning negative feedback into a positive review.<br>

2-Where removing not is harmless:<br>
Topic Classification (e.g., sorting news into Sports, Politics, or Tech) or Search Indexing. In these tasks, words like "not" carry no useful information about the main topic of the document.


### Question 4.6. Stemming and lemmatization
Two ways to bring a word back to a base form:
- stemming cuts the endings with fixed rules (Porter algorithm). It is fast, but the result is not always a real word;
- lemmatization looks the word up in a dictionary (WordNet) and needs its part of speech (`n` noun, `v` verb, `a` adjective). It returns a real word, the lemma.

Run the cell.

**Q4.6.** From the table, give: an example where two words of different meanings get the same stem; an example of a stem that is not a real word; two cases that only the lemmatizer handles correctly.

In [546]:
from nltk.stem import PorterStemmer, WordNetLemmatizer
ps, lem = PorterStemmer(), WordNetLemmatizer()
words = [("studies", "n"), ("studying", "v"), ("university", "n"), ("universe", "n"),
         ("better", "a"), ("ran", "v"), ("happiness", "n"), ("children", "n")]
print(f"{'word':<12}{'Porter stem':<14}{'WordNet lemma':<14}")
for w, pos in words:
    print(f"{w:<12}{ps.stem(w):<14}{lem.lemmatize(w, pos=pos):<14}")

word        Porter stem   WordNet lemma 
studies     studi         study         
studying    studi         study         
university  univers       university    
universe    univers       universe      
better      better        good          
ran         ran           run           
happiness   happi         happiness     
children    children      child         


**Answer Q4.6:**
<br>
1-university and universe:<br>
Both get cut down to the exact same stem: 'univers', even though their meanings are completely different.<br><br><br>
2-A stem that is not a real word:<br>

'studi' (from studies / studying) or 'happi' (from happiness). Neither is an actual English word.<br><br><br>
3-Two cases only the lemmatizer handles correctly (Irregulars):<br>

"better": The lemmatizer correctly finds the root adjective good, while the stemmer leaves it as 'better'.<br>
"ran": The lemmatizer correctly finds the base verb run, while the stemmer fails on irregular past tense and leaves 'ran'.<br>


## Part 5. A complete pipeline

The steps to apply depend on the model:
- a classic model (Bag-of-Words or TF-IDF with Naive Bayes, Logistic Regression, SVM) treats each word as an independent column: every useless variation of a word spreads the statistics, so the text is normalized as much as possible;
- a Transformer (BERT, AraBERT, GPT) was pre-trained on raw text with its own tokenizer: case, punctuation and small words carry information it knows how to use, so only real noise is removed.

### Exercise 5.1. Pipeline for a classic model
Write `preprocess_classic(text)` which returns a list of tokens. Apply in this order:
1. `clean_text`;
2. replace each emoji or emoticon of `EMOJI_MAP` by its value (`<POS>` or `<NEG>`);
3. `expand_contractions`;
4. lowercase;
5. `normalize_ar`;
6. `reduce_elongation` with `keep=1`;
7. `tokenize`;
8. remove the tokens made only of punctuation (the placeholders such as `<url>` are kept);
9. `remove_stopwords` with the three stop-word lists together, keeping the negations.

Note: after step 5, `على` has become `علي` in the text. The stop-word list must be normalized in the same way, otherwise the Arabic stop words will never be found.

In [553]:
ALL_STOPWORDS = {normalize_ar(w) for w in STOPWORDS_EN | STOPWORDS_FR | STOPWORDS_AR}

def preprocess_classic(text):
   
    text = clean_text(text)
    
    
    for emo, tag in EMOJI_MAP.items():
        text = text.replace(emo, tag)
        
   
    text = expand_contractions(text)
    
    
    text = text.lower()
    
    
    text = normalize_ar(text)
    
   
    text = reduce_elongation(text, keep=1)
    
    
    tokens = tokenize(text)
    
   
    tokens = [t for t in tokens if re.search(r"\w", t)]
    
    tokens = remove_stopwords(tokens, ALL_STOPWORDS, keep_negations=True)
    
    return tokens


In [554]:
t1, t4, t8 = (preprocess_classic(COMMENTS[i]) for i in (1, 4, 8))
if not all(isinstance(t, list) for t in (t1, t4, t8)):
    print("Exercise 5.1: FAILED, preprocess_classic must return a list of tokens, not", type(t1).__name__)
else:
    print(t1); print(t4); print(t8)
    check_true("5.1", [
        ("not" in t1, "'not' must be kept in comment 1 (negation)"),
        ("<neg>" in t1 and "<url>" in t1, "comment 1 must contain '<neg>' (from ':(') and '<url>'"),
        ("was" not in t1 and "i" not in t1, "stop words such as 'was' and 'i' must be removed"),
        ("slow" in t1, "'slow' must be kept in comment 1"),
        (all(t == t.lower() for t in t1 + t4 + t8), "all tokens must be in lowercase"),
        (all("http" not in t for t in t1), "a URL is still present"),
        ("ممتازه" in t4, "'ممتاااازة' must become 'ممتازه' (elongation and ta marbuta)"),
        ("الخدمه" in t4, "'الخدمـــة' must become 'الخدمه' (tatweel and ta marbuta)"),
        ("<pos>" in t4, "the emoji 😍 must become '<pos>'"),
        ("ما" in t8 and "ماشي" in t8, "the negations 'ما' and 'ماشي' must be kept"),
        ("في" not in t8, "the stop word 'في' must be removed"),
        (not any(re.fullmatch(r"[^\w<>]+", t) for t in t1 + t4 + t8), "tokens made only of punctuation must be removed"),
    ])

['delivery', 'slow', 'not', 'happy', '<neg>', '<url>']
['الخدمه', 'ممتازه', '<pos>', 'بصح', 'التوصيل', 'lent', 'بزاف', '<user>']
['ما', 'عجبنيش', 'المنتج', 'ماشي', 'كيما', 'الصوره', '<neg>']
Exercise 5.1: OK


<details><summary>Hint 1</summary>

Write the 9 steps as 9 lines: steps 1 to 6 transform `text`, steps 7 to 9 transform `tokens`. Step 2: loop over `EMOJI_MAP.items()` with `text.replace(emo, tag)`. Step 8: keep a token if it contains at least one word character, `re.search(r"\w", t)`.

</details>

<details><summary>Hint 2</summary>

Build the stop-word set once, outside the function: `ALL_STOPWORDS = {normalize_ar(w) for w in STOPWORDS_EN | STOPWORDS_FR | STOPWORDS_AR}` (`|` is the union of two sets). Then `remove_stopwords(tokens, ALL_STOPWORDS)`.

</details>

### Question 5.2. Effect of each step
The cell below computes, for the 12 comments, the vocabulary size (number of distinct tokens) and the total number of tokens after each stage of the pipeline.

**Q5.2.** Which stage reduces the vocabulary the most? Which stages increase the total number of tokens, and why?

In [555]:
if "5.1" not in _PASSED:
    print("Validate exercise 5.1 first.")
else:
    def vocab_size(list_of_token_lists):
        return len({t for tokens in list_of_token_lists for t in tokens})

    def stage_text(c, upto):
        t = c
        if upto >= 2: t = clean_text(t)
        if upto >= 3:
            for emo, tag in EMOJI_MAP.items(): t = t.replace(emo, tag)
            t = expand_contractions(t).lower()
        if upto >= 4: t = reduce_elongation(normalize_ar(t))
        return t

    stages = [
        ("1. raw text, split()",                   [c.split() for c in COMMENTS]),
        ("2. + clean_text",                        [stage_text(c, 2).split() for c in COMMENTS]),
        ("3. + emojis, contractions, lowercase",   [stage_text(c, 3).split() for c in COMMENTS]),
        ("4. + Arabic normalization, elongation",  [stage_text(c, 4).split() for c in COMMENTS]),
        ("5. + tokenize, remove punctuation",      [[t for t in tokenize(stage_text(c, 4)) if re.search(r"\w", t)] for c in COMMENTS]),
        ("6. + stop words (preprocess_classic)",   [preprocess_classic(c) for c in COMMENTS]),
    ]
    sizes = []
    for name, toks in stages:
        sizes.append(vocab_size(toks))
        print(f"{name:<42} vocabulary = {sizes[-1]:>3}   tokens = {sum(len(t) for t in toks):>3}")
    check_true("5.2", [(sizes[-1] < sizes[0], "the final vocabulary should be smaller than the raw one")])

1. raw text, split()                       vocabulary = 106   tokens = 111
2. + clean_text                            vocabulary = 102   tokens = 111
3. + emojis, contractions, lowercase       vocabulary =  91   tokens = 116
4. + Arabic normalization, elongation      vocabulary =  91   tokens = 116
5. + tokenize, remove punctuation          vocabulary =  88   tokens = 117
6. + stop words (preprocess_classic)       vocabulary =  66   tokens =  88
Exercise 5.2: OK


**Answer Q5.2:**
<br>
**1-Stage that reduces the vocabulary the most:**<br>
Stage 6 (Stop words removal):<br>
It reduces the vocabulary the most (from 88 down to 66 distinct words, a drop of 22 words). Filtering out frequent grammatical words cuts down unique terms significantly.<br><br><br>
**2-Stages that increase the total number of tokens, and why:**
<br>
Stage 3 (emojis, contractions, lowercase):<br>
Tokens increased from 111 to 116.<br>
Why: Expanding contractions splits a single word into two separate words (ex: "isn't" becomes "is not", adding an extra token). Also, replacing attached emoticons/emojis with spaced tags like `<POS>` and `<NEG>` creates brand-new individual tokens.<br>
Stage 5 (tokenization):<br>
Tokens increased from 116 to 117.
Why: Proper tokenization detaches punctuation glued to words (for example, in "word!", the punctuation was previously lumped with the word in basic split()). Separating them can uncover additional distinct word tokens.

### Exercise 5.3. Pipeline for a Transformer
Write `preprocess_transformer(text)` which removes only the noise that carries no meaning and returns a string. The tokenizer of the model does the rest.

**Q5.3.** Why do we not lowercase, remove punctuation, remove stop words or stem the text given to a Transformer?

In [559]:
def preprocess_transformer(text):
    return clean_text(text)


In [560]:
check("5.3", preprocess_transformer, [
    (COMMENTS[1], "Delivery was sooooo slow... I'm NOT happy :( <URL>"),
    (COMMENTS[0], "OMG!!! The new phone is AMAZING 😍😍 Tech <USER> <URL>"),
])

Exercise 5.3: OK (2/2 tests)


True

<details><summary>Hint 1</summary>

One of your functions removes HTML, URLs, mentions and extra spaces, and keeps case, punctuation, emojis and stop words.

</details>

**Answer Q5.3:**

<br>**1-Pre-training on raw text:**<br>
Transformers  were pre-trained on billions of real sentences. They expect natural, intact text rather than heavily stripped text.

<br>**2-Capitalization carries meaning:**<br>
Uppercase letters show emotional intensity (e.g. "AMAZING", "NOT happy") and distinguish proper nouns from common words (e.g., "US" the country vs. "us" the pronoun, "Apple" the company vs. "apple" the fruit).<br>

**3-Punctuation and stop words give context:**<br>

Punctuation (!, ?, ...) conveys sentiment, tone, and sentence boundaries.<br>
Stop words and negations provide the grammatical structure that the self-attention mechanism relies on to understand who did what to whom.<br>
No need for stemming:<br>
Transformers already use their own subword tokenizers (like BPE or WordPiece), which handle word variations smoothly without needing destructive rule-based stemming.

## Summary
Run `progress()`, then answer the last questions.

In [562]:
progress()

Validated: 32/32


**Q6.1.** Give the three main stages of preprocessing, in order, with one example of step for each.

**Q6.2.** You build a sentiment classifier for Algerian comments with TF-IDF and Logistic Regression. Give two steps that must not be applied, and explain why.

**Answer Q6.1:**
<br><br><br>
The three main stages of preprocessing, in order, are:
<br><br>
**1-Text Cleaning:**<br>

Removing raw noise that has no value for the model.<br>
ex: Removing HTML tags with BeautifulSoup or replacing URLs with <URL>.<br><br><br>
**2-Tokenization:**<br>

Cutting the text into smaller units (tokens), such as words or subwords.<br>
ex: Splitting a sentence into word tokens using regular expressions or BPE.<br><br><br>
**3-Normalization:**<br>

Converting different variants of the same word into a standard form.<br>
ex: Lowercasing all letters or removing Arabic diacritics (tashkeel).


**Answer Q6.2:**

<br>For sentiment analysis on Algerian comments, two steps that must NOT be applied are:<br><br>

**1-Blindly removing all stop words (specifically negations):**<br>

In Algerian comments, words like "ما", "ماشي", "مش", "pas", and "not" are essential negations.<br>
If we remove them, a negative comment like "ما عجبنيش" (I didn't like it) turns into "عجبني" (I liked it), completely flipping the predicted sentiment.<br><br><br>
**2-Deleting emojis and emoticons:**<br>

In social media comments, emojis like 😍, 😡, 👍, and :( express the user's emotion more directly than words.<br>
Deleting them as punctuation throws away the strongest sentiment signal in the entire text. Instead, they should be mapped to sentiment tags like `<POS>` and `<NEG>`.